# Exercise 5 — SentimentSignal Class

`SentimentSignal` is the stateful wrapper that ties all the pieces together. It follows the same four-method pattern used throughout Section 6 agents: `score` / `score_many` / `signal_from` / `history`. History records every headline and its score — important for auditing an AI-driven trading signal.

In [ ]:
import re

# Gate-safe mock LLM — keyword-based, deterministic, no Ollama required
# Checks only the user message to avoid matching keywords in the system prompt.
def _mock_llm(messages):
    user_text = next(
        (m.get("content", "") for m in messages if m.get("role") == "user"), ""
    ).lower()
    if any(w in user_text for w in ["surge", "rally", "rise", "gain", "bull", "strong"]):
        return "0.75"
    if any(w in user_text for w in ["crash", "fall", "decline", "bear", "weak", "loss"]):
        return "-0.60"
    return "0.10"

BULLISH_HEADLINES = [
    "Tech stocks rally on strong earnings",
    "Markets surge as Fed signals rate pause",
    "S&P 500 gains 2% on positive jobs data",
    "Bull market continues with broad gains",
]
BEARISH_HEADLINES = [
    "Markets crash amid recession fears",
    "Stocks fall sharply on weak economic data",
    "S&P 500 declines on hawkish Fed remarks",
    "Bear market deepens as losses mount",
]
NEUTRAL_HEADLINES = [
    "Markets trade sideways in quiet session",
    "Mixed signals leave investors cautious",
    "Stocks finish flat as investors await data",
]
def parse_score(text):
    """Extract and clamp a float from LLM output. Returns 0.0 if not found."""
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not matches:
        return 0.0
    return max(-1.0, min(1.0, float(matches[0])))

def build_sentiment_prompt(headline):
    return [
        {
            "role": "system",
            "content": (
                "You are a financial news sentiment analyzer. "
                "Score the sentiment from -1.0 (very bearish) to 1.0 (very bullish). "
                "Reply with ONLY a single decimal number. No explanation."
            ),
        },
        {"role": "user", "content": f"Headline: {headline}"},
    ]
def score_headline(headline, llm_fn=None):
    messages = build_sentiment_prompt(headline)
    if llm_fn is not None:
        response = llm_fn(messages)
    else:
        import ollama
        response = ollama.chat(model="llama3.2", messages=messages)["message"]["content"]
    return parse_score(response)
def score_headlines(headlines, llm_fn=None):
    return [score_headline(h, llm_fn) for h in headlines]
def aggregate_sentiment(scores):
    if not scores:
        return 0.0
    return max(-1.0, min(1.0, sum(scores) / len(scores)))

def sentiment_to_signal(sentiment, threshold=0.1):
    return 1 if sentiment > threshold else 0

class SentimentSignal:
    """Stateful news-sentiment signal generator.

    Constructor args:
        llm_fn    : optional injection callable(messages) -> str
        threshold : minimum positive score to go long (default 0.1)

    Methods:
        score(headline)       -> float     — score one headline, record in history
        score_many(headlines) -> list[float] — score list, record each in history
        signal_from(headlines)-> int {0,1} — aggregate+threshold → signal
        history()             -> list[dict] — copy of {headline, score} records
        clear_history()       -> None      — clear history in-place
    """

    def __init__(self, llm_fn=None, threshold=0.1):
        # TODO: store llm_fn, threshold, and initialise _history to empty list
        self._llm_fn = llm_fn
        self._threshold = threshold
        self._history = []

    def score(self, headline):
        # TODO: call score_headline, append {"headline": headline, "score": s}, return s
        return 0.0

    def score_many(self, headlines):
        # TODO: return [self.score(h) for h in headlines]
        return [0.0] * len(headlines)

    def signal_from(self, headlines):
        # TODO: scores = self.score_many(headlines)
        #        return sentiment_to_signal(aggregate_sentiment(scores), self._threshold)
        return 0

    def history(self):
        # TODO: return list(self._history)
        return []

    def clear_history(self):
        # TODO: self._history.clear()
        pass


### Checks

In [ ]:
checks = 0

# 1 — score returns a float and records history
try:
    ss = SentimentSignal(llm_fn=_mock_llm)
    s  = ss.score("Markets rally on earnings")
    assert isinstance(s, float) and -1.0 <= s <= 1.0
    assert len(ss.history()) == 1, f"expected 1 history entry, got {len(ss.history())}"
    assert "headline" in ss.history()[0] and "score" in ss.history()[0]
    checks += 1; print("✅ 1 score returns float and records history entry")
except Exception as e:
    print("❌ 1:", e)

# 2 — score_many records all headlines in history
try:
    ss = SentimentSignal(llm_fn=_mock_llm)
    scores = ss.score_many(BULLISH_HEADLINES)
    assert len(scores) == len(BULLISH_HEADLINES)
    assert len(ss.history()) == len(BULLISH_HEADLINES)
    checks += 1; print("✅ 2 score_many records all headlines in history")
except Exception as e:
    print("❌ 2:", e)

# 3 — signal_from: bullish headlines → 1, bearish → 0
try:
    ss_bull = SentimentSignal(llm_fn=_mock_llm)
    ss_bear = SentimentSignal(llm_fn=_mock_llm)
    bull_sig = ss_bull.signal_from(BULLISH_HEADLINES)
    bear_sig = ss_bear.signal_from(BEARISH_HEADLINES)
    assert bull_sig == 1, f"bullish → expected 1, got {bull_sig}"
    assert bear_sig == 0, f"bearish → expected 0, got {bear_sig}"
    checks += 1; print("✅ 3 signal_from: bullish→1, bearish→0")
except Exception as e:
    print("❌ 3:", e)

# 4 — history returns a copy (not a reference)
try:
    ss = SentimentSignal(llm_fn=_mock_llm)
    ss.score("Test headline")
    h1 = ss.history()
    h1.append({"headline": "injected", "score": 99.0})  # mutate the copy
    h2 = ss.history()
    assert len(h2) == 1, "history() should return a copy, not a reference"
    checks += 1; print("✅ 4 history() returns a copy — internal state protected")
except Exception as e:
    print("❌ 4:", e)

# 5 — clear_history empties history
try:
    ss = SentimentSignal(llm_fn=_mock_llm)
    ss.score_many(BULLISH_HEADLINES)
    assert len(ss.history()) > 0
    ss.clear_history()
    assert len(ss.history()) == 0, "history should be empty after clear_history()"
    checks += 1; print("✅ 5 clear_history empties history")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
